# PaliGemma-3B: V condition, J_PV, J_VT and dJ (D79)

Fixed in **DECISIONS.md D79** before this notebook existed: Qwen's V design
(D75) unchanged, on PaliGemma, with D76/D77's inputs and the H2 layer replay
(D78 (3)) used for patching.

1. **Selection** over all 469 objects: original image and counterfactual
   image, both with the Visual prompt; selected when each image's answer
   follows the image (`v_patching.is_v_conflicting`).
2. **Order and cap:** objects also among PaliGemma's 121 P objects first, then
   seeded random order (seed 0); **at most 244 patched** (D75's cap).
3. **Patching:** source = original image, target = counterfactual image, every
   head, both directions, by **layer replay**: each head recomputes only the
   layers from its own. Checked in section 5 against D76a's batched full
   forward passes on every head of the first object; if that check fails, the
   whole sweep uses the batched full forward passes instead (D79 (3)).
4. **Saves** `pg_v_patching_paligemma2_3b.json` and
   `pg_v_patching_records_paligemma2_3b.jsonl.gz`. Download both into
   `results/` and run `scripts/analyze_pg_v_circuit.py` (it needs at least 60
   objects, D79 (4)).

**Runtime: ~8 h estimated** (~2 min per object) plus ~25 min of selection
and checks; ~16 h if the fallback is used. Each session patches for
`PATCH_BUDGET_H` hours and then saves. To continue, attach the previous
version's output as an input and run again: section 3 prints how much is
already done.

**Untested on the GPU:** `replay.patch_example_replay` is new (the replay
itself ran in H2). Settings: GPU T4, Internet on, `HF_TOKEN` ticked. Run as
**Save Version -> Save & Run All**.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.paligemma": ["CHECKPOINT", "load_paligemma", "build_inputs", "text_geometry",
                                  "FullForwardRunner", "square_image_block",
                                  "last_token_reaches_earlier_positions", "patch_example_batched",
                                  "quiet_processor_warning"],
    "circuit_overlap.patching": ["patch_example", "mean_restoration_grids", "classify_heads",
                                 "capture_last_token_o_proj_inputs"],
    "circuit_overlap.cross_ablation": ["ablate_heads_last_token"],
    "circuit_overlap.replication": ["lietzow_rows", "run_cached"],
    "circuit_overlap.t_patching": ["has_nonfinite", "finite_records"],
    "circuit_overlap.v_patching": ["is_v_conflicting", "v_sweep_order", "V_ROLES", "V_CAP"],
    "circuit_overlap.replay": ["LayerReplay", "patch_example_replay", "patch_jobs",
                               "record_with_activations"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "color_first_token_ids",
                                 "load_notes_config", "native_bf16"],
    "circuit_overlap.heads": ["load_head_set", "format_head_label"],
    "circuit_overlap.metrics": ["jaccard", "containment"],
    "circuit_overlap.provenance": ["result_envelope", "write_result"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("configs/heads_lietzow_paligemma2_3b.json", "results/pg_selection_paligemma2_3b.json",
           "results/pg_replication_patching_paligemma2_3b.json",
           "results/pg_t_patching_paligemma2_3b.json",
           "results/pg_t_patching_records_paligemma2_3b.jsonl.gz",
           "results/split_half_ceiling_p_paligemma2_3b.json", "scripts/analyze_pg_v_circuit.py"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
import inspect as _insp
if not _need:   # D76a additions the name check cannot see
    from circuit_overlap.replication import run_cached as _rc
    from circuit_overlap.paligemma import FullForwardRunner as _ffr
    for _what, _ok in (("run_cached(budget_s=...)", "budget_s" in _insp.signature(_rc).parameters),
                       ("FullForwardRunner(last_only=...)", "last_only" in _insp.signature(_ffr).parameters)):
        if not _ok:
            _need[_what] = ["parameter"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

## 2. Model, geometry, and Lietzow's heads

As `pg_replication.ipynb`. Their heads are only used for the hook check in
section 3; the P set this notebook compares against is the re-discovered one
(identical to theirs, D76).

In [ ]:
import json
from circuit_overlap.paligemma import CHECKPOINT, load_paligemma, quiet_processor_warning, text_geometry
from circuit_overlap.heads import load_head_set, format_head_label
from circuit_overlap.t_screen import PRIOR_PROMPT, VISUAL_PROMPT

quiet_processor_warning()   # the per-call `<image>` nag; section 3 checks the input itself

model, processor = load_paligemma()
COMPUTE_DTYPE = str(model.config.quantization_config.bnb_4bit_compute_dtype).replace("torch.", "")
print(CHECKPOINT, "| compute dtype:", COMPUTE_DTYPE, "(D76 (3): fp16 on a T4)")
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

geo = text_geometry(model)          # raises if o_proj's width != n_heads x head_dim
n_layers, n_heads, head_dim = geo["n_layers"], geo["n_heads"], geo["head_dim"]
print(geo)

HEADS_FILE = f"{REPO_DIR}/configs/heads_lietzow_paligemma2_3b.json"
hs = load_head_set(HEADS_FILE, n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if out of range
hs_raw = json.load(open(HEADS_FILE))
print("their promoting :", [format_head_label(h) for h in hs.promoting])
print("their suppressing:", [format_head_label(h) for h in hs.suppressing])
print("they report:", hs_raw["group_ablation_flip_pct"], "on", hs_raw["n_correctly_conflicting"],
      "objects; single heads", hs.single_head_prior_flip_pct)

## 3. [VERIFY] the same checks as `pg_t_patching.ipynb`

Prompt and image block, `labels`, colour tokens, hooks, attention direction,
last-position logits, and the batched speed (for the fallback path). The T
render check is replaced by one V pair shown by eye.

In [ ]:
import glob, inflect, pathlib, shutil, time
import torch
from collections import Counter
from IPython.display import display
from datasets import load_dataset
from circuit_overlap.paligemma import (
    FullForwardRunner, build_inputs as _pg_inputs, square_image_block,
    last_token_reaches_earlier_positions,
)
from circuit_overlap.patching import capture_last_token_o_proj_inputs
from circuit_overlap.cross_ablation import ablate_heads_last_token
from circuit_overlap.replication import lietzow_rows
from circuit_overlap.t_screen import color_first_token_ids

WORK = pathlib.Path("/kaggle/working/pg_v_patching")
CKPT = WORK / "ckpt"
CKPT.mkdir(parents=True, exist_ok=True)
# Resume: bring back selection.jsonl and records.jsonl from an earlier run
# attached as an input (its notebook output, or the files uploaded as a
# dataset). Found automatically anywhere under /kaggle/input; set RESUME_FROM to
# a folder only if the search picks the wrong files.
RESUME_FROM = None          # e.g. "/kaggle/input/pg-v-patching-ckpt"
for name in ("selection.jsonl", "records.jsonl"):
    if RESUME_FROM:
        found = [f"{RESUME_FROM}/{name}"] if os.path.exists(f"{RESUME_FROM}/{name}") else []
    else:
        found = (sorted(glob.glob(f"/kaggle/input/**/pg_v_patching/ckpt/{name}", recursive=True))
                 or sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)))
    if found and not (CKPT / name).exists():
        shutil.copy(found[0], CKPT / name)
    n_lines = sum(1 for l in open(CKPT / name) if l.strip()) if (CKPT / name).exists() else 0
    print(f"{name}: {n_lines} objects already done" + (f" (from {found[0]})" if found else " (nothing to resume)"))

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
print(len(rows), "objects", dict(dropped), "(the Qwen runs used the same 469)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token on the Gemma tokenizer (D26):", collisions)

def build_inputs(image, prompt):
    return _pg_inputs(processor, image, prompt, model.device)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

def pair_inputs(r):
    """(original-image inputs, counterfactual-image inputs, images), as v_patching.ipynb (D75 (1))."""
    row = ds[r["i"]]
    original = row["original_image"].convert("RGB")
    counterfact = row["counterfact_image"].convert("RGB")
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(original, prompt), build_inputs(counterfact, prompt), (original, counterfact)

VERIFY = {"compute_dtype": COMPUTE_DTYPE, "geometry": geo, "ambiguous_colours": sorted(ambiguous)}
IMAGE_TOKEN_ID = processor.image_token_id      # their get_image_token_id

# [VERIFY] 1-2. the prompt and the image block.
r0 = rows[0]
inp0 = build_inputs(ds[r0["i"]]["counterfact_image"], prompts(r0)[1])
ids0 = inp0["input_ids"][0].tolist()
offset0, side0 = square_image_block(ids0, IMAGE_TOKEN_ID)
text0 = processor.tokenizer.decode(ids0[offset0 + side0 * side0:])
VERIFY["image_block"] = {"offset": offset0, "side": side0, "n_tokens": len(ids0),
                         "pixel_values": list(inp0["pixel_values"].shape)}
VERIFY["prompt_after_image"] = text0
print("inputs:", {k: tuple(v.shape) for k, v in inp0.items()})
print("image block:", VERIFY["image_block"])
print("text after the image:", repr(text0))

# [VERIFY] 7. labels: the first run passed them; this run drops them.
inp0_labels = _pg_inputs(processor, ds[r0["i"]]["counterfact_image"], prompts(r0)[1], model.device,
                         keep_labels=True)
VERIFY["processor_returns_labels"] = "labels" in inp0_labels
if VERIFY["processor_returns_labels"]:
    with torch.no_grad():
        with_labels = model(**inp0_labels, use_cache=False).logits[0, -1].float()
        without = model(**inp0, use_cache=False).logits[0, -1].float()
    VERIFY["labels_max_dlogit"] = float((with_labels - without).abs().max())
    print("logits with vs without labels, max |dlogit|:", VERIFY["labels_max_dlogit"], "(expect ~0)")
    assert VERIFY["labels_max_dlogit"] < 0.05, (
        "labels change the logits (attention mask): the first run's checkpoints are not comparable -- "
        "stop and report this")
    del with_labels, without
del inp0_labels
assert ids0[-1] != IMAGE_TOKEN_ID, "the last token must be text"
assert prompts(r0)[1] in text0, "the prompt is not what the model reads"

# [VERIFY] 3. colour tokens: colliding colours are excluded (D26), and reported.
print("excluded colours (D26):", sorted(ambiguous) or "none")

# [VERIFY] 4. hooks in a full forward pass.
runner0 = FullForwardRunner(model, inp0)
base0 = runner0.logits()
with ablate_heads_last_token(model, [], head_dim):
    same0 = runner0.logits()
with ablate_heads_last_token(model, hs.promoting, head_dim):
    knocked0 = runner0.logits()
with capture_last_token_o_proj_inputs(model) as acts0:
    runner0.logits()
VERIFY["empty_ablation_max_dlogit"] = float((same0 - base0).abs().max())
VERIFY["promoting_ablation_max_dlogit"] = float((knocked0 - base0).abs().max())
print("empty ablation max |dlogit|:", VERIFY["empty_ablation_max_dlogit"], "(expect ~0)")
print("promoting ablation max |dlogit|:", VERIFY["promoting_ablation_max_dlogit"], "(expect > 0)")
assert VERIFY["empty_ablation_max_dlogit"] < 1e-3
assert VERIFY["promoting_ablation_max_dlogit"] > 1e-3
assert sorted(acts0) == list(range(n_layers)) and all(
    acts0[l].shape[-1] == n_heads * head_dim for l in acts0), "captured head outputs have the wrong shape"
del acts0

# [VERIFY] 5. one V pair, by eye: same scene, only the object recoloured.
s0_, c0_, (orig0, cf0) = pair_inputs(r0)
print(f"{r0['name']}: correct {r0['correct']}, counterfactual {r0['attack']!r} | sizes {orig0.size} vs {cf0.size}")
display(orig0.resize((256, 256))); display(cf0.resize((256, 256)))
assert s0_["input_ids"].shape == c0_["input_ids"].shape, "the two images tokenise to different lengths"

# Attention direction (reported, D76 (2)).
VERIFY["last_token_vs_position_0"] = last_token_reaches_earlier_positions(model, inp0, position=0)
VERIFY["last_token_vs_first_text"] = last_token_reaches_earlier_positions(
    model, inp0, position=offset0 + side0 * side0)
for k in ("last_token_vs_position_0", "last_token_vs_first_text"):
    print(k, VERIFY[k])

# [VERIFY] 6. last-position-only logits (runner default) == the full output's last row.
full0 = FullForwardRunner(model, inp0, last_only=False).logits()
VERIFY["logits_to_keep_max_dlogit"] = float((full0 - base0).abs().max())
print("last-only vs full logits, max |dlogit|:", VERIFY["logits_to_keep_max_dlogit"], "(expect ~0)")
assert VERIFY["logits_to_keep_max_dlogit"] < 0.05

# Speed, and the total, before any long loop (CLAUDE.md).
from circuit_overlap.paligemma import _repeat_rows

def _time(fn, reps=2):
    fn(); torch.cuda.synchronize(); t0 = time.time()
    for _ in range(reps):
        fn()
    torch.cuda.synchronize()
    return (time.time() - t0) / reps

step = _time(runner0.logits)
print(f"one forward pass (last-only logits): {step:.2f}s")
VERIFY["forward_s"] = step
per_row = {1: step}
for b in (4, 8, 16):
    try:
        batch_in = _repeat_rows(inp0, b)
        with torch.no_grad():
            per_row[b] = _time(lambda: model(**batch_in, use_cache=False, logits_to_keep=1)) / b
        print(f"batch {b:2d}: {per_row[b]:.2f}s per row")
    except torch.cuda.OutOfMemoryError:
        print(f"batch {b:2d}: out of memory")
        break
    finally:
        batch_in = None
        torch.cuda.empty_cache()
BATCH = min(per_row, key=per_row.get)
VERIFY["per_row_s"], VERIFY["batch_size"] = per_row, BATCH
FALLBACK_S = 2 * step + 2 * n_layers * n_heads * per_row[BATCH]   # D77's method, per object
print(f"batch size {BATCH} for the fallback path: ~{FALLBACK_S / 60:.1f} min per object "
      "(the replay's own estimate comes in section 5)")
del runner0, base0, same0, knocked0, full0

## 4. Selection over all 469 objects (D79 (1), D75 (3))

Full forward passes with last-position logits, the path the replay reproduces.
Checkpointed per object (`ckpt/selection.jsonl`).

In [ ]:
import json
from circuit_overlap.replication import run_cached
from circuit_overlap.v_patching import V_CAP, V_ROLES, is_v_conflicting, v_sweep_order

SEL = json.load(open(f"{REPO_DIR}/results/pg_selection_paligemma2_3b.json"))["result"]
assert {k: sorted(v) for k, v in SEL["color_ids"].items()} == {k: sorted(v) for k, v in color_ids.items()}, \
    "colour tokens differ from the selection run: the tokenizer changed"
row_by_i = {r["i"]: r for r in rows}

def select_object(r):
    src_in, tgt_in, (orig, cf) = pair_inputs(r)
    out = {"name": r["name"], "attack": r["attack"],
           "size_original": list(orig.size), "size_counterfact": list(cf.size)}
    if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
        return {"status": "shape_mismatch", **out}
    logits_orig = FullForwardRunner(model, src_in).logits()
    logits_cf = FullForwardRunner(model, tgt_in).logits()
    if not (torch.isfinite(logits_orig).all() and torch.isfinite(logits_cf).all()):
        return {"status": "numerics_failed", **out}       # plan risk 7
    top1_orig, top1_cf = int(logits_orig.argmax()), int(logits_cf.argmax())
    out.update(top1_original=top1_orig, top1_counterfactual=top1_cf)
    ok = is_v_conflicting(r, top1_orig, top1_cf, color_ids, ambiguous)
    return {"status": "selected" if ok else "not_conflicting", **out}

selection = run_cached(CKPT / "selection.jsonl", rows, select_object, progress_every=50)
assert len(selection) == len(rows), "selection did not finish: run the notebook again to resume it"
print(Counter(s["status"] for s in selection))
selected_top1 = {s["i"]: (s["top1_original"], s["top1_counterfactual"])
                 for s in selection if s["status"] == "selected"}
selected = sorted(selected_top1)

P_FILE = f"{REPO_DIR}/results/pg_replication_patching_paligemma2_3b.json"
p_ids = [r["i"] for r in json.load(open(P_FILE))["result"]["records"]]
order = v_sweep_order(selected, p_ids, seed=0, cap=V_CAP)
todo = [row_by_i[i] for i in order]
n_shared = len(set(order) & set(p_ids))
print(f"{len(selected)} selected of {len(rows)}; {len(todo)} to patch (cap {V_CAP}); "
      f"the first {n_shared} are also P objects (Lietzow's intersection subset)")
print("roles:", V_ROLES)

## 5. Patching sweep (D79 (3))

First the check D79 (3) fixes: on the first V object, every head in both
directions by layer replay vs `patch_example_batched` (D76a's path, which
produced P and T). Same answer pair and max |dlogit| < 0.5, or the sweep uses
`patch_example_batched`. Then, per object, each run's unablated replay must
match its full pass within 0.1, or that object is patched the slow way.

In [ ]:
from circuit_overlap.paligemma import patch_example_batched
from circuit_overlap.replay import patch_example_replay
from circuit_overlap.t_patching import has_nonfinite

PATCH_BUDGET_H = 11.0     # hours of patching this session (Kaggle stops a session at 12 h)
REPLAY_TOL = 0.1          # D79 (3): per-object unablated replay vs full pass
GRIDS = ("p2v_logit_prior", "p2v_logit_visual", "v2p_logit_prior", "v2p_logit_visual")

# [VERIFY] replay == batched full forward passes, every head, both directions, first V object.
src1, tgt1, _ = pair_inputs(todo[0])
torch.cuda.synchronize(); t0 = time.time()
fast = patch_example_replay(model, src1, tgt1, head_dim, batch_size=n_heads)
torch.cuda.synchronize(); replay_s = time.time() - t0
t0 = time.time()
slow = patch_example_batched(model, src1, tgt1, head_dim, batch_size=BATCH)
torch.cuda.synchronize(); batched_s = time.time() - t0
assert slow is not None and fast is not None, "the first V object gives the same answer on both images"
diff = max(abs(a - b) for k in GRIDS for ra, rb in zip(slow[k], fast[k]) for a, b in zip(ra, rb))
same_pair = (slow["next_token_idx_prior"], slow["next_token_idx_visual"]) == \
            (fast["next_token_idx_prior"], fast["next_token_idx_visual"])
USE_REPLAY = same_pair and diff < 0.5
VERIFY["replay_vs_batched"] = {"object": todo[0]["name"], "n_patches": 2 * n_layers * n_heads,
                               "same_answer_pair": same_pair, "max_dlogit": diff,
                               "replay_check": fast["replay_check"], "replay_s": replay_s,
                               "batched_s": batched_s, "use_replay": USE_REPLAY}
print(VERIFY["replay_vs_batched"])
per_object = replay_s if USE_REPLAY else batched_s
print(("layer replay" if USE_REPLAY else "REPLAY CHECK FAILED -> batched full forward passes (D79 (3))")
      + f": ~{per_object:.0f} s per object, ~{len(todo) * per_object / 3600:.1f} h for {len(todo)} objects")
del slow, fast, src1, tgt1

def sweep_object(r):
    src_in, tgt_in, _ = pair_inputs(r)
    if src_in["input_ids"].shape != tgt_in["input_ids"].shape:
        return {"status": "shape_mismatch", "name": r["name"]}
    # Roles (V_ROLES): source = original image, target = counterfactual image.
    method = "replay" if USE_REPLAY else "batched"
    record = patch_example_replay(model, src_in, tgt_in, head_dim, batch_size=n_heads) if USE_REPLAY else None
    if record is not None and record["replay_check"] > REPLAY_TOL:
        method, record = "batched_fallback", None
    if method != "replay":
        record = patch_example_batched(model, src_in, tgt_in, head_dim, batch_size=BATCH)
    if record is None:                                     # argmax tokens coincide
        return {"status": "same_argmax", "name": r["name"], "method": method}
    top1 = (record["next_token_idx_prior"], record["next_token_idx_visual"])
    out = {"name": r["name"], "top1_original": top1[0], "top1_counterfactual": top1[1], "method": method}
    if top1 != selected_top1[r["i"]]:
        return {"status": "selection_drift", **out}
    for key, value in record.items():
        if isinstance(value, list):
            record[key] = [[round(x, 4) for x in row] for row in value]
    full = {"status": "patched", **out, **record}
    if has_nonfinite(full):
        full["status"] = "numerics_failed"                 # plan risk 7
    return full

sweep = run_cached(CKPT / "records.jsonl", todo, sweep_object, progress_every=5,
                   budget_s=PATCH_BUDGET_H * 3600)
print(Counter(s["status"] for s in sweep), Counter(s.get("method") for s in sweep),
      f"-- {len(sweep)} of {len(todo)} V objects done")

## 6. Classify and save (after every session)

Saves a complete snapshot of everything done so far, so each version's output
can be analysed. The verdict comes from `scripts/analyze_pg_v_circuit.py` on
the downloaded files; the preliminary run below is the same script.

In [ ]:
import gzip, sys
from circuit_overlap.patching import mean_restoration_grids, classify_heads
from circuit_overlap.t_patching import finite_records
from circuit_overlap.metrics import jaccard, containment
from circuit_overlap.provenance import result_envelope, write_result

patched, nonfinite = finite_records([s for s in sweep if s["status"] == "patched"])
checks = [s["replay_check"] for s in sweep if "replay_check" in s]
summary = {"n_objects": len(rows), "selection_status": dict(Counter(s["status"] for s in selection)),
           "n_selected": len(selected), "cap": V_CAP, "n_todo": len(todo), "n_done": len(sweep),
           "n_scored": len(patched), "sweep_status": dict(Counter(s["status"] for s in sweep)),
           "methods": dict(Counter(s.get("method") for s in sweep)),
           "replay_check_max": max(checks) if checks else None, "nonfinite_ids": nonfinite,
           "order_seed": 0, "n_shared_with_p": n_shared, "roles": V_ROLES, "verify": VERIFY,
           "selection": selection, "records_file": "pg_v_patching_records_paligemma2_3b.jsonl.gz"}
if patched:
    p2v, v2p, n_used = mean_restoration_grids(patched)
    cls = classify_heads(p2v, v2p, sigma=2.0)
    v_prom = set(cls["promoting"])
    p_prom = {tuple(h) for h in json.load(open(P_FILE))["result"]["promoting"]}
    t_prom = {tuple(h) for h in json.load(open(f"{REPO_DIR}/results/pg_t_patching_paligemma2_3b.json"))["result"]["promoting"]}
    print(f"{n_used} objects scored | V promoting:", sorted(format_head_label(h) for h in v_prom))
    print(f"PRELIMINARY J_PV {jaccard(p_prom, v_prom):.3f} | J_VT {jaccard(v_prom, t_prom):.3f} | "
          f"J_PT {jaccard(p_prom, t_prom):.3f} | containment V in P {containment(v_prom, p_prom):.2f}")
    summary.update({
        "p2v_grid": p2v.round(6).tolist(), "v2p_grid": v2p.round(6).tolist(),
        "pc1_scores": cls["pc1_scores"].round(6).tolist(), "pc1_direction": cls["pc1_direction"],
        "pc1_mean": cls["pc1_mean"], "pc1_std": cls["pc1_std"], "sigma": cls["sigma"],
        "promoting": [list(h) for h in sorted(v_prom)],
        "suppressing": [list(h) for h in sorted(cls["suppressing"])]})

print(write_result(WORK / "pg_v_patching_paligemma2_3b.json", result_envelope(
    "pg_v_patching", summary, model="PaliGemma2-3B-mix-448", checkpoint=CHECKPOINT, seed=0,
    prompt=f"visual={VISUAL_PROMPT!r}; original vs counterfactual image", n=len(patched),
    repo_dir=REPO_DIR, decision="D79", quantisation="4-bit nf4, double quant",
    compute_dtype=COMPUTE_DTYPE, gpu=torch.cuda.get_device_name(0), image_tokens=side0 * side0,
    method="exact layer replay per patched head, checked against batched full forward passes (D79 (3)); "
           "Lietzow PCA +/-2 sigma; source=original image, target=counterfactual image")))
with gzip.open(WORK / "pg_v_patching_records_paligemma2_3b.jsonl.gz", "wt") as f:
    for s in sweep:
        f.write(json.dumps(s) + "\n")
for name in ("pg_v_patching_paligemma2_3b.json", "pg_v_patching_records_paligemma2_3b.jsonl.gz"):
    print(f"  {name:46s} {(WORK / name).stat().st_size / 1e6:6.2f} MB")

if len(patched) >= 60:
    sys.path.insert(0, f"{REPO_DIR}/scripts")
    from analyze_pg_v_circuit import main as d79_v
    print("\nPRELIMINARY (the committed script on this session's files):\n")
    d79_v(str(WORK / "pg_v_patching_records_paligemma2_3b.jsonl.gz"))
else:
    print(f"\n{len(patched)} objects patched: fewer than 60 (D79 (4)), no preliminary verdict yet")
print(f"\n{len(sweep)} of {len(todo)} done. To continue: attach this version's output and run again.")